# Human-in-the-loop tool confirmation

How `@requires_confirmation` works: a tool declares it needs a human **yes**, and the permission gate pauses before it runs — even under a bypass mode. This notebook shows the gate directly, then a real agent asking for approval.

In [1]:
# Load the LOCAL checkout (repo root is two levels up from notebooks/tool_confirmation).
import sys, pathlib
repo = pathlib.Path.cwd().parent.parent
if str(repo) not in sys.path: sys.path.insert(0, str(repo))
import shipit_agent
print('shipit-agent', shipit_agent.__version__, 'from', shipit_agent.__file__)
assert 'site-packages' not in shipit_agent.__file__, 'restart kernel — installed pkg loaded'

shipit-agent 1.7.1 from /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/shipit_agent/__init__.py


## 1. The decorator + the gate (no LLM)

`PermissionEngine.check()` precedence: **deny > @requires_confirmation floor > mode (bypass/plan) > allow > ask > callback**. A confirmed tool returns `ASK` even in bypass.

In [2]:
from shipit_agent.tools import requires_confirmation
from shipit_agent.permissions import PermissionEngine
from shipit_agent.tools.base import ToolOutput

@requires_confirmation('Deletes a path permanently — cannot be undone.', impact='irreversible')
class DeleteTool:
    name = 'delete_path'
    description = 'Delete a file or directory.'
    def schema(self):
        return {'type':'function','function':{'name':self.name,'description':self.description,
                'parameters':{'type':'object','properties':{'path':{'type':'string'}},'required':['path']}}}
    def run(self, context=None, **kw):
        return ToolOutput(text=f"deleted {kw.get('path')}", metadata={})

# Even in BYPASS mode (which would allow everything), the tool floors it to ASK:
engine = PermissionEngine(mode='bypass')
decision = engine.check('delete_path', {'path': '/tmp/report.pdf'}, tool=DeleteTool)
print('decision :', decision.decision.value)
print('reason   :', decision.reason)

decision : ask
reason   : Deletes a path permanently — cannot be undone.


### Conditional confirmation — only the dangerous call pauses
A `when(arguments)` predicate lets a cheap call run free.

In [3]:
@requires_confirmation('Large transfer needs sign-off.', impact='irreversible',
                       when=lambda a: a.get('amount', 0) >= 10_000)
class WireTransfer:
    name = 'wire_transfer'
    def schema(self):
        return {'type':'function','function':{'name':self.name,'parameters':{'type':'object',
                'properties':{'amount':{'type':'number'},'to':{'type':'string'}},'required':['amount','to']}}}
    def run(self, context=None, **kw): return ToolOutput(text='sent', metadata={})

eng = PermissionEngine(mode='bypass')
print('$5      ->', eng.check('wire_transfer', {'amount':5,'to':'x'}, tool=WireTransfer).decision.value)
print('$50,000 ->', eng.check('wire_transfer', {'amount':50_000,'to':'x'}, tool=WireTransfer).decision.value)

$5      -> allow
$50,000 -> ask


## 2. A real agent asking for approval

Wire a `permission_callback(name, args) -> PermissionResult | None` into the Agent. When the model calls the confirmed tool, the gate consults your callback — that's your human-in-the-loop. Here we auto-approve after printing the prompt; in a UI you'd draw a card.

In [5]:
import os, json as _json
# HITL demo needs any provider. Set VERTEX_SA_KEY (or GOOGLE_APPLICATION_CREDENTIALS)
# to your service-account JSON path — never hardcode a path/key in the notebook.
VKEY = os.environ.get('VERTEX_SA_KEY') or os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
# assert VKEY, 'Set VERTEX_SA_KEY=/path/to/service-account.json before running this cell.'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = VKEY
os.environ['VERTEXAI_PROJECT'] = _json.load(open(VKEY))['project_id']
os.environ['VERTEXAI_LOCATION'] = os.environ.get('VERTEXAI_LOCATION', 'us-central1')
from shipit_agent.llms.factory import build_llm_from_settings
llm = build_llm_from_settings({'provider':'vertex','model':'vertex_ai/gemini-2.5-flash'}, load_env=False)
llm

TypeError: str expected, not NoneType

In [ ]:
from shipit_agent import Agent
from shipit_agent.permissions import PermissionResult, PermissionDecision

def human_approval(name, args):
    # This is your HITL surface. Return ALLOW / DENY (optionally with edited args),
    # or None to defer to the rest of the policy.
    print(f'\n  🔔 CONFIRM: the agent wants to run {name}({args})')
    print('     → auto-approving for this demo (in a UI, a human clicks here)')
    return PermissionResult(PermissionDecision.ALLOW, reason='approved by human')

agent = Agent(llm=llm, tools=[DeleteTool()], permission_callback=human_approval,
              auto_use_skills=False, auto_project_memory=False, skill_source=None, max_iterations=4)

result = agent.run('Delete the file /tmp/old_report.pdf using the delete_path tool.')
print('\nANSWER:', result.output)

## How it works, end to end

1. The model emits a tool call.
2. Before executing, the runtime calls `PermissionEngine.check(name, args, tool)`.
3. A **hard deny** always wins. Otherwise, if the tool was decorated with `@requires_confirmation` (and its `when` predicate applies), the gate consults your `permission_callback` — the human-in-the-loop.
4. `ALLOW` → the tool runs (with any edited arguments). `DENY` → the tool is skipped and the model is told. `ASK` with no callback → an approval event a UI can draw a card from.
5. It's a **floor**: no bypass/allow mode can silently skip a tool that demands confirmation.